In [2]:
import pandas as pd

In [3]:

llm_sim_gain = pd.read_csv("../parquet/llm_gen_experiment/data/llm_gen/simulatability_analysis/1.7B_llm_heart_disease_1200_predictor_answers_simulatability_analysis.csv")
reg_sim_gain = pd.read_csv("../parquet/llm_gen_experiment/data/reg_gen/simulatability_analysis/1.7B_reg_heart_disease_1200_predictor_answers_simulatability_analysis.csv")

In [6]:
df = pd.concat([llm_sim_gain,reg_sim_gain])

In [7]:
df

,model,total,with_explanation_accuracy,without_explanation_accuracy,simulatability_gain,gain_ci_lower,gain_ci_upper,with_ci_lower,with_ci_upper,without_ci_lower,...,recall_ci_lower,recall_ci_upper,recall_without,recall_without_ci_lower,recall_without_ci_upper,diff_pct,both_correct,both_wrong,only_with_correct,only_without_correct
0,Qwen/Qwen3-1.7B_True,1099,84.440400,79.981802,4.458599,1.909048,7.006689,81.403247,87.364477,76.250597,...,86.825173,92.865809,82.857143,78.603411,86.643855,20.382166,823,115,105,56
0,Qwen/Qwen3-1.7B_True,1200,84.666667,78.916667,5.750000,2.995909,8.517701,81.984235,87.318506,76.209854,...,86.659820,91.878516,80.450358,77.454685,83.524168,18.583333,891,128,125,56


## Problem 1 — Underpowered
SE on the between-arm difference (~1.9pp) exceeds the effect (~1.3pp). z ≈ 0.67, p ≈ 0.5. The instrument is coarser than what you're trying to measure.

**Fix:** Pair the arms — same reference questions in both, one counterfactual per method per question. Question difficulty becomes a shared shock and cancels in the subtraction instead of compounding. Cuts SE 30–50% at zero compute cost.

---

## Problem 2 — Disjoint question sets
The arms drew different questions, so any NSG gap is confounded with item difficulty (`diff_pct` already differs: 20.4% vs 18.6%). This is a *validity* problem — more data would not fix it.

**Fix:** Same as Problem 1. Pairing kills the confound and shrinks the SE in one move. This is why it's the single highest-value change you can make.

---

## Problem 3 — NSG-difference can't isolate faithfulness
Your explanations are identical across arms — same model, same questions. Only the *yardstick* changed. So a difference tells you the **generators** differ, which is already expected: NSG falls with counterfactual distance (Fig. 17) and LLM generators produce incoherent items (Table 7). Both depress NSG for reasons unrelated to faithfulness.

**Fix:** Reframe the claim. The defensible version is *"NSG is sensitive to counterfactual-generation method, which complicates cross-study comparison"* — not *"LLM-gen explanations are less faithful."* Support it by reporting distance and coherence per arm.

---

## Problem 4 — Single predictor
Your CIs capture item noise only. They say nothing about whether another predictor flips the sign — and the paper shows single predictors do exactly that.

**Fix:** Use the existing `--average` flag. `analyze_simulatability_averaged` already pools predictors and bootstraps correctly. Free if the predictions exist; otherwise state it as a limitation.

<!-- predictors to tyr -->

Practical suggestion. Sub-4B is fine if you verify non-degeneracy. Candidates worth trying, different families:

Qwen3-4B — but this is your reference model's family, so exclude or use --exclude-self-predictors family
gemma-3-4b-it — the paper shows Gemma-3-4B performing well as a reference model
Llama-3.2-3B-Instruct
Phi-3.5-mini (3.8B)



Experiment incomplete without these. These help adress problem 3
Distance measurement per arm. Hamming/edit distance distributions, side by side. If the LLM arm drifts further, that alone predicts lower NSG (paper Fig. 17).
Coherence audit. Hand-check ~50 LLM-generated counterfactuals for the implausible-combination failure (Table 7). Report the rate.

Moreover get ai to explain purpose of each of the analyiss files

